# State gaming source inventory

This notebook reads the researched inventory and the latest coverage table.
There is **no downloading** here.

Coverage statuses you will see later:

| Status | Meaning |
| --- | --- |
| `ok` | Official revenue (or handle+tax) rows were collected |
| `partial` | Some periods collected; others missing or unpublished |
| `blocked` | Access control, captcha, or no clean online/retail split |
| `not_publicly_available` | Official site does not publish a usable series |
| `combined_only` | Official report mixes products or channels |
| `pdf_only_not_yet_parsed` | Public PDFs exist; parser not promoted yet |
| `legal_not_reporting` | Authorized but no operating reports yet |
| `on_premises_only` / `location_based_mobile` / `annual_only` | Special cases kept out of a simple online comparison |

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "config").exists() and (ROOT.parent / "config").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.common import project_root
from variant_gaming.storage import connect_readonly

ROOT = project_root()
database_file = "data/staging/gaming_nationwide.sqlite"  # explicitly select another snapshot if needed
database_path = ROOT / database_file
print("Read-only database:", database_path)
inventory = pd.read_csv(ROOT / "config" / "state_gaming_source_inventory.csv")
print(len(inventory), "inventory rows")
inventory[["jurisdiction", "state_code", "vertical", "recommended_wave"]].head(12)

## Columns

| Column | Meaning |
| --- | --- |
| `jurisdiction` | State or district name |
| `state_code` | Postal code |
| `vertical` | Product (`online_sports_betting` or `online_casino` in the primary table) |
| `official_landing_url` | Regulator page |
| `typical_format` | CSV, Excel, PDF, dashboard |
| `public_granularity` | Operator vs statewide |
| `recommended_wave` | Collection order (1 first) |
| `status_note` | Research caveats |

In [ ]:
wave_counts = (
    inventory.groupby("recommended_wave", dropna=False)
    .size()
    .reset_index(name="source_count")
    .sort_values("recommended_wave")
)
wave_counts

## Wave 1 starter sources

In [ ]:
inventory.loc[inventory["recommended_wave"] == 1, [
    "jurisdiction", "state_code", "vertical", "official_landing_url", "status_note"
]].sort_values(["state_code", "vertical"])

## Coverage already recorded in SQLite (if present)

Select `database_file` in the setup cell. Inspection defaults to staging and uses a read-only connection; an absent database is reported without creating one.


In [ ]:
coverage = pd.DataFrame()
if database_path.exists():
    conn = connect_readonly(database_path)
    try:
        coverage = pd.read_sql_query('SELECT state_code, vertical, status, earliest_period, latest_period, normalized_row_count FROM source_coverage ORDER BY state_code, vertical', conn)
    finally:
        conn.close()
else:
    print("Selected database does not exist; inspection skipped. No database was created.")
coverage
